# ValidMind + Databricks Quickstart

This notebook validates that the ValidMind Library works correctly within a Databricks Collaborative Notebook environment. It demonstrates:

- Installing and initializing the ValidMind Library
- Loading data from a Unity Catalog table via Spark
- Training a simple classification model
- Running ValidMind tests and sending results to the ValidMind Platform

## Before you begin

You will need:
1. A running Databricks workspace with Unity Catalog enabled
2. A ValidMind account with a registered model
3. Your ValidMind API credentials (API key, API secret, model identifier)

To get your credentials: log in to ValidMind → **Model Inventory** → select your model → **Getting Started** → **Copy snippet to clipboard**.

> **Note:** If you don't have a UC table ready, this notebook includes a fallback that generates synthetic data so you can still validate the full workflow.

## Step 1 — Install the ValidMind Library

Run this cell first. Databricks requires a Python restart after `%pip install`.

In [ ]:
%pip install -q validmind

In [ ]:
# Restart Python kernel to pick up newly installed packages
dbutils.library.restartPython()

## Step 2 — Verify installation

In [ ]:
import importlib.metadata
version = importlib.metadata.version('validmind')
print(f'ValidMind Library version: {version}')
print('Installation successful!')

## Step 3 — Initialize the ValidMind Library

Replace the placeholder values below with your actual credentials from the ValidMind Platform.

For development, use `https://api.dev.vm.validmind.ai/api/v1/tracking` as the `api_host`.
For production, use `https://app.prod.validmind.ai/api/v1/tracking/tracking`.

In [ ]:
import validmind as vm

# ---------------------------------------------------------------------------
# Credentials — injected by ValidMind Platform when run via "Run Tests",
# or replace the fallback values to run this notebook manually.
# ---------------------------------------------------------------------------
try:
    api_host   = dbutils.widgets.getAll().get("vm_api_host", "https://api.dev.vm.validmind.ai/api/v1/tracking")
    api_key    = dbutils.widgets.getAll().get("vm_api_key", "YOUR_API_KEY")
    api_secret = dbutils.widgets.getAll().get("vm_api_secret", "YOUR_API_SECRET")
    model_cuid = dbutils.widgets.getAll().get("vm_model_cuid", "YOUR_MODEL_CUID")
except NameError:
    # dbutils is not available — running outside Databricks
    api_host   = "https://api.dev.vm.validmind.ai/api/v1/tracking"
    api_key    = "YOUR_API_KEY" # replace with your API key
    api_secret = "YOUR_API_SECRET" # replace with your API secret
    model_cuid = "YOUR_MODEL_CUID" # replace with your model CUID

vm.init(
    api_host=api_host,
    api_key=api_key,
    api_secret=api_secret,
    model=model_cuid,
)


## Step 4 — Load data from your linked Databricks table

Instead of querying Databricks directly, this notebook loads data through ValidMind.
ValidMind fetches and syncs the Unity Catalog table data when you create a binding in
**Settings → Integrations → Databricks**, so the same dataset is available here via the
tracking API — no Spark session or direct UC credentials needed.

**Prerequisites:**
1. A Databricks integration configured in ValidMind Settings
2. A `table` binding created for this model (link a Unity Catalog table to this model)
3. At least one successful sync (the initial sync triggers automatically on binding creation)

If no binding exists yet, set `USE_SYNTHETIC_FALLBACK = True` to run with generated data.

In [ ]:
import requests
import pandas as pd
from validmind import api_client as _vm_client

# Set to True only if you don't have a Databricks table binding set up yet
USE_SYNTHETIC_FALLBACK = False

# ---------------------------------------------------------------------------
# Load from ValidMind — uses the linked Databricks table binding for this model
# ---------------------------------------------------------------------------
if not USE_SYNTHETIC_FALLBACK:
    _api_host = _vm_client.get_api_host()  # same host as vm.init()
    _headers  = _vm_client._get_api_headers()

    _response = requests.get(
        f"{_api_host}/integrations/dataset",
        headers=_headers,
        timeout=30,
    )

    if _response.status_code == 200:
        _data = _response.json()
        TABLE_NAME    = _data.get("table_name", "unknown")
        TARGET_COLUMN = "target"  # <-- update if your table uses a different column name
        row_data      = _data.get("row_data", [])

        if not row_data:
            raise RuntimeError(
                f"Binding found for table '{TABLE_NAME}' but row_data is empty. "
                "The sync may still be in progress — wait a moment and re-run this cell."
            )

        df = pd.DataFrame(row_data)

        if TARGET_COLUMN not in df.columns:
            raise ValueError(
                f"Column '{TARGET_COLUMN}' not found in synced data. "
                f"Available columns: {list(df.columns)}. "
                "Update TARGET_COLUMN above to match your table's target column."
            )

        print(f"Loaded {len(df):,} rows, {len(df.columns)} columns from {TABLE_NAME}")
        print(f"Last synced: {_data.get('last_synced_at', 'unknown')}")
        print(f"Target distribution: {df[TARGET_COLUMN].value_counts().to_dict()}")
        display(df.head())

    elif _response.status_code == 404:
        raise RuntimeError(
            "No active Databricks table binding found for this model.\n\n"
            "To fix:\n"
            "  1. Go to ValidMind → Settings → Integrations → Databricks\n"
            "  2. Open the model binding browser and select a Unity Catalog table\n"
            "  3. Wait ~30 seconds for the initial sync to complete\n"
            "  4. Re-run this cell\n\n"
            "Or set USE_SYNTHETIC_FALLBACK = True above to continue with generated data."
        )
    else:
        raise RuntimeError(
            f"Unexpected error loading dataset from ValidMind: "
            f"{_response.status_code} — {_response.text}"
        )

In [ ]:
# ---------------------------------------------------------------------------
# Synthetic data fallback — runs when USE_SYNTHETIC_FALLBACK = True
# Uses the Bank Customer Churn dataset pattern from ValidMind examples
# ---------------------------------------------------------------------------
if USE_SYNTHETIC_FALLBACK:
    import numpy as np
    from sklearn.datasets import make_classification

    np.random.seed(42)
    X, y = make_classification(
        n_samples=1000,
        n_features=10,
        n_informative=6,
        n_redundant=2,
        random_state=42,
    )
    feature_names = [
        "credit_score", "age", "tenure", "balance",
        "num_products", "has_credit_card", "is_active_member",
        "estimated_salary", "geography_encoded", "gender_encoded",
    ]
    df = pd.DataFrame(X, columns=feature_names)
    df["target"] = y
    TARGET_COLUMN = "target"
    TABLE_NAME    = "synthetic"

    print(f"Using synthetic dataset: {len(df):,} rows, {len(df.columns)} columns")
    print(f"Target distribution: {df[TARGET_COLUMN].value_counts().to_dict()}")
    display(df.head())

## Step 5 — Prepare train/test split

In [ ]:
from sklearn.model_selection import train_test_split

feature_columns = [c for c in df.columns if c != TARGET_COLUMN]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f'Train set: {len(train_df):,} rows')
print(f'Test set:  {len(test_df):,} rows')
print(f'Features:  {feature_columns}')

## Step 6 — Train a simple model

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(train_df[feature_columns], train_df[TARGET_COLUMN])

train_accuracy = model.score(train_df[feature_columns], train_df[TARGET_COLUMN])
test_accuracy  = model.score(test_df[feature_columns],  test_df[TARGET_COLUMN])

print(f'Train accuracy: {train_accuracy:.4f}')
print(f'Test accuracy:  {test_accuracy:.4f}')

## Step 7 — Register datasets and model with ValidMind

In [ ]:
vm_train_ds = vm.init_dataset(
    dataset=train_df,
    input_id="train_dataset",
    target_column=TARGET_COLUMN,
)

vm_test_ds = vm.init_dataset(
    dataset=test_df,
    input_id="test_dataset",
    target_column=TARGET_COLUMN,
)

vm_model = vm.init_model(
    model=model,
    input_id="gradient_boosting_model",
)

print('Datasets and model registered with ValidMind.')

## Step 8 — Assign predictions to datasets

In [ ]:
vm_train_ds.assign_predictions(model=vm_model)
vm_test_ds.assign_predictions(model=vm_model)

print('Predictions assigned.')

## Step 9 — Run individual tests

These tests validate that results render correctly in the notebook and are sent to the ValidMind Platform.

In [ ]:
# Dataset statistics — validates data documentation capability
result = vm.tests.run_test(
    "validmind.data_validation.DatasetDescription",
    inputs={"dataset": vm_train_ds},
)
result.log()

In [ ]:
# Class imbalance check
result = vm.tests.run_test(
    "validmind.data_validation.ClassImbalance",
    inputs={"dataset": vm_train_ds},
)
result.log()

In [ ]:
# Confusion matrix — validates model performance visualization
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.ConfusionMatrix",
    inputs={"dataset": vm_test_ds, "model": vm_model},
)
result.log()

In [ ]:
# ROC curve
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.ROCCurve",
    inputs={"dataset": vm_test_ds, "model": vm_model},
)
result.log()

In [ ]:
# Feature importance
result = vm.tests.run_test(
    "validmind.model_validation.sklearn.FeatureImportance",
    inputs={"dataset": vm_train_ds, "model": vm_model},
)
result.log()

## Step 10 — Run the full test suite

This runs the complete classifier documentation suite and sends all results to ValidMind in one call.

> This is the primary validation that results can be sent from a Databricks notebook environment.

In [ ]:
test_suite_result = vm.run_test_suite(
    "classifier_full_suite",
    inputs={
        "dataset": vm_test_ds,
        "model": vm_model,
        "train_dataset": vm_train_ds,
        "test_dataset": vm_test_ds,
    },
)
print('Full test suite completed and results sent to ValidMind Platform.')

## Step 11 — Verify results on the platform

1. Go to [ValidMind Platform](https://app.prod.validmind.ai) (or your local instance)
2. Navigate to **Model Inventory** → your model
3. Open the **Documentation** tab
4. Confirm that test results from this notebook appear

**Expected results visible on platform:**
- Dataset Description table
- Class Imbalance chart
- Confusion Matrix
- ROC Curve
- Feature Importance chart
- Full classifier suite results

---

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `ModuleNotFoundError` after install | Re-run the `dbutils.library.restartPython()` cell |
| `ConnectionError` on `vm.init()` | Workspace may block outbound traffic — check network policy or use a cluster with internet access |
| `401 Unauthorized` on `vm.init()` | API key/secret are incorrect — copy credentials fresh from the platform |
| `numpy` version conflict | Pin with `%pip install -q validmind "numpy>=1.23,<2.0.0"` |
| `404` on dataset load | No Databricks table binding found — create one in Settings → Integrations → Databricks, then wait for sync |
| `row_data is empty` after binding created | Initial sync is still running — wait ~30 seconds and re-run Step 4 |
| Wrong columns / target not found | Update `TARGET_COLUMN` in Step 4 to match the actual target column in your UC table |
| Want to test without a binding | Set `USE_SYNTHETIC_FALLBACK = True` in Step 4 |